# Mini Data Annotation Tool (Google Colab Version)

Project portofolio untuk menunjukkan pengalaman **Human-Centered Data** (data annotation / labeling / prompt evaluation).

Hasil labeling akan disimpan permanen ke Google Drive, jadi tidak hilang walau session Colab restart.

**Urutan menjalankan:**
1. Jalankan cell 'Mount Google Drive' lalu ikuti link login
2. Jalankan cell 'Setup dataset'
3. Jalankan cell 'Proses Labeling' dan mulai label review satu per satu
4. (Opsional) Jalankan cell 'Hitung Inter-Rater Agreement' kalau sudah ada 2 annotator

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Semua file project akan disimpan di folder ini di Google Drive kamu
PROJECT_DIR = '/content/drive/MyDrive/data_annotation_tool'
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/output', exist_ok=True)

print('Folder project siap di:', PROJECT_DIR)

## 2. Install dependency

In [ ]:
!pip install -q scikit-learn pandas

## 3. Setup Dataset

Kalau file `sample_reviews.csv` belum ada di Drive, cell ini akan otomatis membuatnya. Kalau sudah ada (misal kamu lanjutkan project lama), cell ini akan pakai yang sudah ada, tidak menimpa.

In [ ]:
import pandas as pd

DATA_PATH = f'{PROJECT_DIR}/data/sample_reviews.csv'
OUTPUT_PATH = f'{PROJECT_DIR}/output/labeled_reviews.csv'

if not os.path.exists(DATA_PATH):
    reviews = [
        "Barangnya bagus banget, sesuai deskripsi dan pengiriman cepat!",
        "Kecewa berat, barang datang rusak dan penjual susah dihubungi.",
        "Lumayan lah, sesuai harga. Tidak ada yang spesial.",
        "Kualitas produk sangat memuaskan, akan order lagi.",
        "Barang tidak sesuai gambar, ukurannya beda jauh.",
        "Pengiriman lama tapi produknya oke.",
        "Sangat merekomendasikan toko ini, pelayanan ramah dan cepat.",
        "Produk cepat rusak, baru dipakai 2 hari sudah rusak.",
        "Standar aja sih, tidak buruk tapi juga tidak istimewa.",
        "Packing rapi, barang sampai dengan selamat, terima kasih!",
        "Warna produk berbeda dari foto, agak mengecewakan.",
        "Harga sebanding dengan kualitas, worth it!",
        "Respon penjual lambat, tapi barangnya bagus.",
        "Tidak akan beli lagi di toko ini, pengalaman buruk.",
        "Produk sesuai ekspektasi, tidak ada komplain.",
        "Kualitas biasa saja untuk harga segini.",
        "Sangat puas dengan pembelian ini, top!",
        "Barang cacat produksi, minta refund tapi lama prosesnya.",
        "Oke lah, sesuai fungsinya.",
        "Pelayanan mengecewakan dan barang telat 2 minggu.",
    ]
    df = pd.DataFrame({
        'review_id': range(1, len(reviews) + 1),
        'review_text': reviews,
    })
    df.to_csv(DATA_PATH, index=False)
    print('Dataset baru dibuat dan disimpan ke Drive.')
else:
    df = pd.read_csv(DATA_PATH)
    print('Dataset sudah ada, memuat dari Drive.')

print(f'Jumlah review: {len(df)}')
df.head()

## 4. Proses Labeling Manual

Jalankan cell ini, masukkan nama kamu sebagai annotator, lalu beri label untuk tiap review yang muncul.

- `1` = Positif
- `2` = Negatif
- `3` = Netral
- `q` = Simpan & berhenti (bisa lanjut lagi nanti, review yang sudah dilabel tidak akan diulang)

In [ ]:
VALID_LABELS = {'1': 'positif', '2': 'negatif', '3': 'netral'}

def load_existing_labels(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    return pd.DataFrame(columns=['review_id', 'review_text', 'label', 'annotator'])

def ask_label(review_text):
    print('\n' + '-' * 60)
    print(f'Review: {review_text}')
    print('-' * 60)
    print('1 = Positif | 2 = Negatif | 3 = Netral | q = Simpan & keluar')
    while True:
        choice = input('Jawaban kamu: ').strip().lower()
        if choice == 'q':
            return 'quit'
        if choice in VALID_LABELS:
            return VALID_LABELS[choice]
        print('Input tidak valid, masukkan 1, 2, 3, atau q.')

annotator_name = input('Masukkan nama annotator kamu (misal: budi): ').strip()

labeled_df = load_existing_labels(OUTPUT_PATH)
already_labeled_ids = labeled_df[labeled_df['annotator'] == annotator_name]['review_id'].tolist()

new_rows = []
for _, row in df.iterrows():
    if row['review_id'] in already_labeled_ids:
        continue
    label = ask_label(row['review_text'])
    if label == 'quit':
        print('\nMenyimpan progres dan keluar...')
        break
    new_rows.append({
        'review_id': row['review_id'],
        'review_text': row['review_text'],
        'label': label,
        'annotator': annotator_name,
    })

if new_rows:
    new_df = pd.DataFrame(new_rows)
    result_df = pd.concat([labeled_df, new_df], ignore_index=True)
    result_df.to_csv(OUTPUT_PATH, index=False)
    print(f'\n{len(new_rows)} review berhasil dilabel dan disimpan ke Google Drive.')
else:
    print('\nTidak ada review baru yang dilabel (mungkin semua sudah dilabel oleh annotator ini).')

## 5. Hitung Inter-Rater Agreement (Cohen's Kappa)

Jalankan cell ini setelah minimal 2 annotator berbeda sudah melabel dataset yang sama (bisa kamu sendiri jalankan 2x dengan nama beda untuk testing, atau ajak teman).

In [ ]:
from sklearn.metrics import cohen_kappa_score

result_df = pd.read_csv(OUTPUT_PATH)
annotators = result_df['annotator'].unique().tolist()

if len(annotators) < 2:
    print('Baru ada 1 annotator. Minta 1 orang lagi label dataset yang sama dulu.')
else:
    print(f'Annotator ditemukan: {annotators}')
    a1, a2 = annotators[0], annotators[1]

    df1 = result_df[result_df['annotator'] == a1][['review_id', 'label']].rename(columns={'label': f'label_{a1}'})
    df2 = result_df[result_df['annotator'] == a2][['review_id', 'label']].rename(columns={'label': f'label_{a2}'})
    merged = pd.merge(df1, df2, on='review_id', how='inner')

    if len(merged) == 0:
        print(f'{a1} dan {a2} belum melabel review yang sama.')
    else:
        raw_agreement = (merged[f'label_{a1}'] == merged[f'label_{a2}']).mean()
        kappa = cohen_kappa_score(merged[f'label_{a1}'], merged[f'label_{a2}'])

        print(f'Jumlah review yang dilabel kedua annotator: {len(merged)}')
        print(f'Raw agreement ({a1} vs {a2}): {raw_agreement:.2%}')
        print(f"Cohen's Kappa: {kappa:.3f}")

        if kappa < 0:
            print('Interpretasi: Tidak ada kesepakatan.')
        elif kappa < 0.20:
            print('Interpretasi: Kesepakatan sangat rendah.')
        elif kappa < 0.40:
            print('Interpretasi: Kesepakatan rendah.')
        elif kappa < 0.60:
            print('Interpretasi: Kesepakatan sedang.')
        elif kappa < 0.80:
            print('Interpretasi: Kesepakatan tinggi.')
        else:
            print('Interpretasi: Kesepakatan sangat tinggi.')

## 6. (Opsional) Visualisasi Distribusi Label

In [ ]:
import matplotlib.pyplot as plt

result_df = pd.read_csv(OUTPUT_PATH)
label_counts = result_df['label'].value_counts()

plt.figure(figsize=(6,4))
label_counts.plot(kind='bar', color=['#4CAF50', '#F44336', '#9E9E9E'])
plt.title('Distribusi Label Hasil Annotasi')
plt.xlabel('Label')
plt.ylabel('Jumlah Review')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()